In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy==1.26.4", "--force-reinstall"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "scipy>=1.11.0,<1.14.0",
    "scikit-learn>=1.3.0",
    "pandas>=2.0.0,<3.0.0",
])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "accelerate", "peft", "bitsandbytes", "trl", "evaluate",
    "matplotlib", "seaborn",
])

print("Done. Restart kernel and run from Cell 2.")

In [1]:
# Cell 2 - Imports and basic setup
import os
import re
import gc
import json
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# Cell 3 - Find dataset
CANDIDATE_PATHS = [
    Path("/kaggle/input/datasets/sakhadib/bangladesh-legal-acts-dataset/balanced_1000.csv"),
    Path("/kaggle/input/bangladesh-legal-acts-dataset/balanced_1000.csv"),
    Path("/kaggle/working/balanced_1000.csv"),
    Path("balanced_1000.csv"),
    Path("./data/balanced_1000.csv"),
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    matches = list(Path(".").rglob("balanced_1000.csv"))
    if matches:
        DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find balanced_1000.csv. Upload it to Kaggle input or put it in /kaggle/working."
    )

df = pd.read_csv(DATA_PATH)
print("Using dataset:", DATA_PATH)
print("Shape:", df.shape)
display(df.head(3))

Using dataset: /kaggle/input/datasets/sakhadib/bangladesh-legal-acts-dataset/balanced_1000.csv
Shape: (3160, 4)


,act_title,year,section,govt_system
0,"The Railways Act, 1890",1890,"69. Every passenger by the railway, shall, on ...",british Colonial
1,"The Tanks Improvement Act, 1939 (Bengal Act).",1939,25. (1) During the period of possession all di...,british Colonial
2,"The Waqfs Ordinance, 1962 (East Pakistan Ordin...",1962,"14. The salaries, and the terms and conditions...",Militery Rule


In [3]:
# Cell 4 - Shuffle and split
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

n_total = len(df)
n_train = int(0.8 * n_total)
n_temp  = n_total - n_train
n_valid = n_temp // 2

train_df_raw = df.iloc[:n_train].copy()
valid_df_raw = df.iloc[n_train:n_train + n_valid].copy()
test_df_raw  = df.iloc[n_train + n_valid:].copy()

print("Raw Train:", train_df_raw.shape)
print("Raw Valid:", valid_df_raw.shape)
print("Raw Test :", test_df_raw.shape)

Raw Train: (2528, 4)
Raw Valid: (316, 4)
Raw Test : (316, 4)


In [4]:
# Cell 5 - Convert to QA format
def convert_to_qa(df):
    rows = []

    for _, r in df.iterrows():
        act_title = str(r["act_title"]).strip() if pd.notna(r.get("act_title")) else "this law"
        year = str(r["year"]).strip() if pd.notna(r.get("year")) else ""
        section_text = str(r["section"]).strip() if pd.notna(r.get("section")) else ""

        if not section_text:
            continue

        act_name = f"{act_title} ({year})" if year else act_title
        question = f"What does {act_name} say in this provision?"
        context = [{"id": "C1", "text": section_text}]
        answer = section_text

        rows.append({
            "question": question,
            "context": context,
            "answer": answer,
            "citations": ["C1"],
            "language": "en",
        })

    return pd.DataFrame(rows)

train_df = convert_to_qa(train_df_raw)
valid_df = convert_to_qa(valid_df_raw)
test_df  = convert_to_qa(test_df_raw)

print("QA Train:", train_df.shape)
print("QA Valid:", valid_df.shape)
print("QA Test :", test_df.shape)
display(train_df.head(2))

QA Train: (2497, 5)
QA Valid: (309, 5)
QA Test : (311, 5)


,question,context,answer,citations,language
0,What does [বাংলাদেশ মেরিটাইম ইউনিভার্সিটি] আইন...,"[{'id': 'C1', 'text': '৫৯। বিশ্ববিদ্যালয়ের বা...",৫৯। বিশ্ববিদ্যালয়ের বার্ষিক প্রতিবেদন সিন্ডিক...,[C1],en
1,"What does The Finance Act, 1974 (1974) say in ...","[{'id': 'C1', 'text': '5. In the Amusement Tax...","5. In the Amusement Tax Act, 1922 (Ben. Act V ...",[C1],en


In [5]:
# Cell 6 - Prompt helpers
SYSTEM_PROMPT = """You are a legal QA assistant.
Answer ONLY from the provided context.

Rules:
1. Do not invent facts.
2. If the answer is not fully supported by the context, say: "Insufficient evidence in provided context."
3. Cite supporting context IDs inline like [C1], [C2].
4. Match the language of the user's question.
"""

def normalize_context(ctx):
    out = []
    if isinstance(ctx, list):
        for i, item in enumerate(ctx, start=1):
            if isinstance(item, dict):
                out.append({
                    "id": item.get("id", f"C{i}"),
                    "text": item.get("text", "")
                })
            else:
                out.append({
                    "id": f"C{i}",
                    "text": str(item)
                })
    return out

def build_context_block(ctx_items):
    ctx_items = normalize_context(ctx_items)
    return "\n\n".join([f"[{x['id']}] {x['text']}" for x in ctx_items])

def build_user_prompt(example):
    context_block = build_context_block(example["context"])
    return f"""Question:
{example['question']}

Context:
{context_block}

Provide a grounded answer with inline citations.
"""

def safe_citations(citations):
    if isinstance(citations, list):
        return [str(x) for x in citations]
    if isinstance(citations, str):
        cleaned = citations.strip().strip("[]")
        if not cleaned:
            return []
        return [x.strip().strip("'").strip('"') for x in cleaned.split(",")]
    return []

def append_inline_citations(answer, citations):
    cits = safe_citations(citations)
    ans = str(answer).strip()

    if not cits:
        return ans

    inline = " ".join([f"[{c}]" for c in cits])
    if inline in ans:
        return ans
    return f"{ans} {inline}"

def build_messages(example):
    assistant_text = append_inline_citations(example["answer"], example.get("citations", []))
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(example)},
        {"role": "assistant", "content": assistant_text},
    ]

In [6]:
# Cell 7 - Load tokenizer
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [7]:
# Cell 8 - Convert dataframe to SFT format
def dataframe_to_sft_df(df, tokenizer):
    rows = []
    for _, row in df.iterrows():
        ex = row.to_dict()
        messages = build_messages(ex)
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        rows.append({"text": text})
    return pd.DataFrame(rows)

train_sft = dataframe_to_sft_df(train_df, tokenizer)
valid_sft = dataframe_to_sft_df(valid_df, tokenizer)

print("Train SFT:", train_sft.shape)
print("Valid SFT:", valid_sft.shape)
display(train_sft.head(2))

Train SFT: (2497, 1)
Valid SFT: (309, 1)


,text
0,<|im_start|>system\nYou are a legal QA assista...
1,<|im_start|>system\nYou are a legal QA assista...


In [8]:
# Cell 9 - Dataset class
class TextSFTDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=768):
        self.texts = df["text"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = item["input_ids"].clone()
        item["labels"][item["attention_mask"] == 0] = -100
        return item

MAX_LENGTH = 768

train_ds = TextSFTDataset(train_sft, tokenizer, max_length=MAX_LENGTH)
valid_ds = TextSFTDataset(valid_sft, tokenizer, max_length=MAX_LENGTH)

print("Dataset objects ready.")
print("Train size:", len(train_ds))
print("Valid size:", len(valid_ds))

Dataset objects ready.
Train size: 2497
Valid size: 309


In [9]:
# Cell 10 - Build 4-bit Qwen + LoRA model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def build_qwen_lora_model():
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model.config.use_cache = False

    lora_config = LoraConfig(
        r=4,
        lora_alpha=8,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = get_peft_model(model, lora_config)
    return model

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = build_qwen_lora_model()
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [10]:
# Cell 11 - Training arguments
OUTPUT_DIR = Path("/kaggle/working/week4_best_qwen_retrain")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "trainer_output"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=2,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    fp16=not torch.cuda.is_available(),
    bf16=torch.cuda.is_available(),
    dataloader_pin_memory=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
# Cell 12 - Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
)

train_result = trainer.train()
print(train_result)

Step,Training Loss,Validation Loss
100,0.781202,0.725308
200,0.782862,0.675817
300,0.723213,0.662961
313,0.723213,0.662890


TrainOutput(global_step=313, training_loss=0.8904329854459427, metrics={'train_runtime': 8550.2546, 'train_samples_per_second': 0.292, 'train_steps_per_second': 0.037, 'total_flos': 1.5089546048569344e+16, 'train_loss': 0.8904329854459427, 'epoch': 1.0})


In [13]:
# Cell 13 - Save LoRA adapter and tokenizer
ADAPTER_SAVE_DIR = OUTPUT_DIR / "best_week4_qwen_adapter"
ADAPTER_SAVE_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(ADAPTER_SAVE_DIR))
tokenizer.save_pretrained(str(ADAPTER_SAVE_DIR))

print("Saved adapter to:", ADAPTER_SAVE_DIR)
print("Files:")
for p in sorted(ADAPTER_SAVE_DIR.iterdir()):
    print("-", p.name)

Saved adapter to: /kaggle/working/week4_best_qwen_retrain/best_week4_qwen_adapter
Files:
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- tokenizer.json
- tokenizer_config.json


In [14]:
# Cell 14 - Reload adapter to verify it works
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model_for_inference = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

loaded_model = PeftModel.from_pretrained(base_model_for_inference, str(ADAPTER_SAVE_DIR))
loaded_model.eval()

loaded_tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_SAVE_DIR), trust_remote_code=True)
if loaded_tokenizer.pad_token is None:
    loaded_tokenizer.pad_token = loaded_tokenizer.eos_token

print("Adapter reloaded successfully.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Adapter reloaded successfully.


In [15]:
# Cell 15 - Smoke test generation
def generate_answer(tokenizer, model, example, max_new_tokens=180):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(example)},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    with torch.no_grad():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = model_inputs["input_ids"].shape[1]
    output_tokens = generated[0][input_len:]
    prediction = tokenizer.decode(output_tokens, skip_special_tokens=True).strip()
    return prediction

sample_example = test_df.iloc[0].to_dict()
sample_pred = generate_answer(loaded_tokenizer, loaded_model, sample_example)

print("QUESTION:")
print(sample_example["question"])
print("\nPREDICTION:")
print(sample_pred)
print("\nGOLD:")
print(append_inline_citations(sample_example["answer"], sample_example["citations"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION:
What does আয়কর আইন, ২০২৩ (2023) say in this provision?

PREDICTION:
�পকর কমিশনার উপ-ধারা (২) এর অধীন নোটিশ প্রদান করিতে পারিবেন না।

Provide a grounded answer with inline citations.

GOLD:
২৫৮। (১) আদালতের আদেশের মাধ্যমে বা অন্য কোনোভাবে অবলুপ্ত (wound up) কোনো প্রাইভেট কোম্পানির লিকুইডেটর, তাহার লিকুইডেটর হিসাবে নিযুক্ত হইবার ৩০ (ত্রিশ) দিনের মধ্যে তাহার নিযুক্তির বিষয়ে যে উপকর কমিশনারের অধীন উক্ত কোম্পানির কর নির্ধারণের অধিক্ষেত্র রহিয়াছে তাহার নিকট নোটিশ প্রদান করিবেন।(২) উপকর কমিশনার উপ-ধারা (১) এর অধীন নোটিশ প্রাপ্তির তারিখ হইতে ৩ (তিন) মাসের মধ্যে, তাহার নিজস্ব বিবেচনায় আবশ্যকীয় তথ্যাদি তলব অথবা প্রয়োজনীয় তদন্ত সম্পন্ন করে, কোম্পানি কর্তৃক উক্ত সময়ে অথবা উক্ত সময়ের পরে প্রদেয় কর পরিশোধের জন্য তাহার মতে যেই পরিমাণ অর্থের প্রয়োজন হইবে সেই সম্পর্কে লিকুইডেটরকে অবগত করিবেন।(৩) উপ-ধারা (২) এর অধীন অবগত হইবার পর লিকুইডেটর নোটিশে উল্লিখিত করের সমপরিমাণ অর্থ পৃথক করিয়া রাখিবেন এবং উক্ত অর্থ পৃথক করিবার পূর্বে, কোম্পানি অবলুপ্তির তারিখে কোম্পানি কর্তৃক প্রদেয় কর পরি

In [16]:
# Cell 16 - Zip adapter for easy download
import shutil

zip_base = "/kaggle/working/best_week4_qwen_adapter"
zip_path = shutil.make_archive(zip_base, 'zip', str(ADAPTER_SAVE_DIR))

print("Created zip:", zip_path)

Created zip: /kaggle/working/best_week4_qwen_adapter.zip
